# Graph-PRefLexOR-4B-Inpainting: standalone Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lamm-mit/graph-preflexor-grpo/blob/main/Notebooks/Graph_PRefLexOR_4B_Inpainting_Colab.ipynb)

This notebook runs `lamm-mit/Graph-PRefLexOR-4B-Inpainting`. Every graph-canvas, prompt, parsing,
validation, exact-scoring, visualization, and generation helper used below is
defined directly in the notebook.

The notebook constructs and tests all six trained graph corruption modes:

1. `prior_empty`
2. `fixed_nodes_only`
3. `missing_edges`
4. `partial_subgraph`
5. `wrong_relations`
6. `extra_edges`

**Runtime:** select **Runtime → Change runtime type → GPU**. An A100 is best
for BF16. The default 4-bit option is intended to fit smaller Colab GPUs.


## 1. Install runtime dependencies

This installs only standard inference/notebook packages. It does **not**
install the Graph-PRefLexOR repository.


In [ ]:
get_ipython().run_line_magic(
    "pip",
    "-q install -U 'transformers>=5.10.1' accelerate bitsandbytes "
    "huggingface_hub safetensors sentencepiece protobuf pandas matplotlib "
    "networkx tqdm",
)


## 2. Imports, authentication, and runtime settings

If Hugging Face requests authentication, add an `HF_TOKEN` secret in Colab
(key icon in the left sidebar). The model is public but remains subject to the
Gemma license and access terms.

Keep `USE_4BIT=True` for T4/L4-class GPUs. Set it to `False` before the model
loading cell when using an A100 with sufficient memory and you want BF16
weights. The original rollout limit was 4,096 tokens; reduce
`MAX_NEW_TOKENS` if a smaller GPU runs out of KV-cache memory.


In [ ]:
import html
import json
import math
import random
import re
import time
from collections import Counter
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Mapping, Optional

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, Markdown, display
from huggingface_hub import login
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from google.colab import userdata

    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
except ImportError:
    HF_TOKEN = None

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    print("No HF_TOKEN secret found; attempting anonymous public-model access.")

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."

MODEL_ID = "lamm-mit/Graph-PRefLexOR-4B-Inpainting"
USE_4BIT = True
ENABLE_THINKING = True
MAX_PROMPT_TOKENS = 4096
MAX_NEW_TOKENS = 4096
TEMPERATURE = 0.0
TOP_P = 1.0
SEED = 11

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
GPU_MAJOR = torch.cuda.get_device_capability(0)[0]
COMPUTE_DTYPE = torch.bfloat16 if GPU_MAJOR >= 8 else torch.float16

print(f"GPU: {GPU_NAME} ({GPU_MEMORY_GB:.1f} GiB)")
print(f"Compute dtype: {COMPUTE_DTYPE}; 4-bit weights: {USE_4BIT}")


## 3. Standalone graph schema and graph-canvas functions

These functions reproduce the graph identity and canvas conventions used by
training. Node identity is the exact `id`; edge identity is the exact
`(source, relation, target)` triple. Optional payload fields are retained.


In [ ]:
GRAPH_COMPLETION_MODES = (
    "prior_empty",
    "fixed_nodes_only",
    "missing_edges",
    "partial_subgraph",
    "wrong_relations",
    "extra_edges",
)


def edge_key(edge: Mapping[str, Any]) -> str:
    """Return the exact semantic identity of an edge."""

    return "\t".join(
        str(edge.get(name, "")).strip()
        for name in ("source", "relation", "target")
    )


def compact_json(value: Mapping[str, Any]) -> str:
    return json.dumps(
        dict(value), ensure_ascii=False, sort_keys=True, separators=(",", ":")
    )


def validate_graph_schema(graph: Any) -> list[str]:
    """Validate required graph fields without deleting optional payloads."""

    errors: list[str] = []
    if not isinstance(graph, dict):
        return ["graph is not a JSON object"]
    for collection in ("nodes", "edges"):
        if collection not in graph:
            errors.append(f"missing required {collection!r} array")
        elif not isinstance(graph[collection], list):
            errors.append(f"{collection!r} is not an array")
    if errors:
        return errors
    for index, node in enumerate(graph["nodes"]):
        if not isinstance(node, dict):
            errors.append(f"nodes[{index}] is not an object")
        elif not str(node.get("id", "")).strip():
            errors.append(f"nodes[{index}] has no non-empty id")
    for index, edge in enumerate(graph["edges"]):
        if not isinstance(edge, dict):
            errors.append(f"edges[{index}] is not an object")
            continue
        for key in ("source", "relation", "target"):
            if not str(edge.get(key, "")).strip():
                errors.append(f"edges[{index}] has no non-empty {key}")
    return errors


def audit_raw_graph(graph: Any) -> dict[str, float]:
    """Report schema, duplicate-object, and dangling-edge defects."""

    schema_errors = validate_graph_schema(graph)
    if not isinstance(graph, dict):
        return {
            "schema_valid": 0.0,
            "duplicate_nodes": 0.0,
            "duplicate_edges": 0.0,
            "dangling_edges": 0.0,
            "structural_validity": 0.0,
        }
    nodes = graph.get("nodes", []) if isinstance(graph.get("nodes"), list) else []
    edges = graph.get("edges", []) if isinstance(graph.get("edges"), list) else []
    node_ids = [
        str(node.get("id", "")).strip()
        for node in nodes
        if isinstance(node, dict)
    ]
    edge_keys = [edge_key(edge) for edge in edges if isinstance(edge, dict)]
    node_counts = Counter(node_ids)
    edge_counts = Counter(edge_keys)
    duplicate_nodes = sum(
        max(0, count - 1) for key, count in node_counts.items() if key
    )
    duplicate_edges = sum(
        max(0, count - 1)
        for key, count in edge_counts.items()
        if key.strip("\t")
    )
    node_set = {node_id for node_id in node_ids if node_id}
    dangling_edges = sum(
        str(edge.get("source", "")).strip() not in node_set
        or str(edge.get("target", "")).strip() not in node_set
        for edge in edges
        if isinstance(edge, dict)
    )
    schema_valid = not schema_errors
    return {
        "schema_valid": float(schema_valid),
        "duplicate_nodes": float(duplicate_nodes),
        "duplicate_edges": float(duplicate_edges),
        "dangling_edges": float(dangling_edges),
        "structural_validity": float(
            schema_valid
            and duplicate_nodes == 0
            and duplicate_edges == 0
            and dangling_edges == 0
        ),
    }


def canonicalize_graph(graph: Mapping[str, Any]) -> dict[str, Any]:
    """Canonicalize ordering while preserving optional object payloads."""

    if not isinstance(graph, Mapping):
        raise TypeError("graph must be a mapping")
    node_by_id: dict[str, dict[str, Any]] = {}
    for node in graph.get("nodes", []) or []:
        if not isinstance(node, Mapping):
            continue
        node_id = str(node.get("id", "")).strip()
        if not node_id:
            continue
        clean = {str(key): value for key, value in node.items()}
        clean["id"] = node_id
        node_by_id.setdefault(node_id, clean)
    edge_by_key: dict[str, dict[str, Any]] = {}
    for edge in graph.get("edges", []) or []:
        if not isinstance(edge, Mapping):
            continue
        clean = {str(key): value for key, value in edge.items()}
        for key in ("source", "relation", "target"):
            clean[key] = str(clean.get(key, "")).strip()
        if not all(clean[key] for key in ("source", "relation", "target")):
            continue
        edge_by_key.setdefault(edge_key(clean), clean)
    return {
        "nodes": sorted(node_by_id.values(), key=lambda item: str(item["id"])),
        "edges": sorted(
            edge_by_key.values(),
            key=lambda item: (
                str(item["source"]),
                str(item["relation"]),
                str(item["target"]),
                compact_json(item),
            ),
        ),
    }


def render_graph_canvas(
    graph: Mapping[str, Any],
    *,
    fixed_node_ids: Iterable[str] = (),
    fixed_edge_keys: Iterable[str] = (),
) -> str:
    """Render the exact textual canvas consumed by the model."""

    fixed_nodes = set(map(str, fixed_node_ids))
    fixed_edges = set(map(str, fixed_edge_keys))
    canonical = canonicalize_graph(graph)
    lines = ["<graph_canvas>", "<nodes>"]
    for node in canonical["nodes"]:
        prefix = "[FIXED] " if str(node["id"]) in fixed_nodes else ""
        lines.append(f"{prefix}N {compact_json(node)}")
    lines.extend(["</nodes>", "<edges>"])
    for edge in canonical["edges"]:
        prefix = "[FIXED] " if edge_key(edge) in fixed_edges else ""
        lines.append(f"{prefix}E {compact_json(edge)}")
    lines.extend(["</edges>", "</graph_canvas>"])
    return "\n".join(lines)


GRAPH_COMPLETION_INSTRUCTION = """Infer the complete scientifically appropriate graph from the condition and incomplete or corrupted graph canvas below.

Rules:
- Preserve every [FIXED] node and edge exactly, including all payload fields.
- Make only changes permitted by the stated task and corruption mode.
- Add missing scientific content when needed.
- Remove spurious content when required.
- Correct wrong relations when required.
- Return the complete corrected graph, not a patch or a list of operations.
- Preserve every node and edge field present in the graph schema, including optional metadata.
- Put the final JSON object inside <answer>...</answer>.
- The JSON object must contain nodes and edges arrays.
- Emit nothing after </answer>.

You may reason using the model's native thinking channel. Only the final answer block is scored.
"""


def build_graph_completion_user_prompt(
    canvas: str,
    *,
    condition: Optional[str] = None,
    mode: Optional[str] = None,
) -> str:
    condition_line = f"Condition:\n{condition.strip()}\n\n" if condition else ""
    mode_line = f"Corruption mode: {mode}\n\n" if mode else ""
    return (
        f"{GRAPH_COMPLETION_INSTRUCTION}\n"
        f"{condition_line}{mode_line}{canvas.strip()}"
    )


def apply_graph_completion_chat_template(
    tokenizer: Any,
    user_prompt: str,
    *,
    enable_thinking: bool = True,
) -> str:
    """Apply Gemma's chat template with a compatibility fallback."""

    messages = [{"role": "user", "content": user_prompt}]
    kwargs = {
        "tokenize": False,
        "add_generation_prompt": True,
    }
    try:
        return tokenizer.apply_chat_template(
            messages, enable_thinking=enable_thinking, **kwargs
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kwargs)


## 4. Strict answer extraction and exact graph metrics

Evaluation below is deterministic and symbolic. It does not use embeddings
or an LLM judge. Node/edge order is ignored after canonicalization, while
IDs, relation strings, endpoints, fixed objects, and payloads are exact.


In [ ]:
ANSWER_RE = re.compile(r"<answer>(.*?)</answer>", re.DOTALL)


@dataclass
class AnswerExtraction:
    raw_output: str
    reasoning: str = ""
    answer_text: Optional[str] = None
    graph: Optional[dict[str, Any]] = None
    canonical_graph: Optional[dict[str, Any]] = None
    errors: list[str] = field(default_factory=list)
    answer_tags_valid: bool = False
    termination_valid: bool = False
    json_valid: bool = False
    schema_valid: bool = False

    @property
    def valid(self) -> bool:
        return bool(
            self.answer_tags_valid
            and self.termination_valid
            and self.json_valid
            and self.schema_valid
        )


def extract_final_answer(output: Any) -> AnswerExtraction:
    """Extract only the final complete `<answer>` block."""

    raw = str(output or "")
    result = AnswerExtraction(raw_output=raw)
    matches = list(ANSWER_RE.finditer(raw))
    if not matches:
        result.errors.append(
            "missing closing </answer> tag"
            if "<answer>" in raw
            else "missing <answer> block"
        )
        return result
    match = matches[-1]
    result.reasoning = raw[: match.start()]
    result.answer_text = match.group(1).strip()
    result.answer_tags_valid = True
    result.termination_valid = not raw[match.end() :].strip()
    if not result.termination_valid:
        result.errors.append("non-whitespace content follows </answer>")
    try:
        parsed = json.loads(result.answer_text)
    except Exception as exc:
        result.errors.append(f"answer JSON parse failed: {exc}")
        return result
    if not isinstance(parsed, dict):
        result.errors.append("answer JSON is not an object")
        return result
    result.graph = parsed
    result.json_valid = True
    schema_errors = validate_graph_schema(parsed)
    if schema_errors:
        result.errors.extend(schema_errors)
        return result
    result.schema_valid = True
    result.canonical_graph = canonicalize_graph(parsed)
    return result


def ratio(numerator: int, denominator: int, *, empty: float = 1.0) -> float:
    return empty if denominator == 0 else numerator / denominator


def prf(matched: int, predicted: int, target: int) -> tuple[float, float, float]:
    if predicted == 0 and target == 0:
        return 1.0, 1.0, 1.0
    precision = ratio(matched, predicted, empty=0.0)
    recall = ratio(matched, target, empty=1.0)
    f1 = (
        0.0
        if precision + recall == 0
        else 2 * precision * recall / (precision + recall)
    )
    return precision, recall, f1


def node_map(graph: Mapping[str, Any]) -> dict[str, dict[str, Any]]:
    return {str(node["id"]): dict(node) for node in graph.get("nodes", [])}


def edge_map(graph: Mapping[str, Any]) -> dict[str, dict[str, Any]]:
    return {edge_key(edge): dict(edge) for edge in graph.get("edges", [])}


def endpoint_map(graph: Mapping[str, Any]) -> dict[tuple[str, str], set[str]]:
    result: dict[tuple[str, str], set[str]] = {}
    for edge in graph.get("edges", []):
        endpoint = (str(edge["source"]), str(edge["target"]))
        result.setdefault(endpoint, set()).add(str(edge["relation"]))
    return result


def compute_graph_metrics(
    prediction: Mapping[str, Any],
    target: Mapping[str, Any],
    source_graph: Mapping[str, Any],
    *,
    fixed_node_ids: Optional[list[str]] = None,
    fixed_edge_keys: Optional[list[str]] = None,
) -> dict[str, float]:
    """Compute the exact graph metrics used for the six-mode demonstration."""

    raw_audit = audit_raw_graph(prediction)
    pred = canonicalize_graph(prediction)
    gold = canonicalize_graph(target)
    source = canonicalize_graph(source_graph)
    p_nodes, g_nodes, x_nodes = node_map(pred), node_map(gold), node_map(source)
    p_edges, g_edges, x_edges = edge_map(pred), edge_map(gold), edge_map(source)

    node_precision, node_recall, node_f1 = prf(
        len(set(p_nodes) & set(g_nodes)), len(p_nodes), len(g_nodes)
    )
    edge_precision, edge_recall, edge_f1 = prf(
        len(set(p_edges) & set(g_edges)), len(p_edges), len(g_edges)
    )

    pred_endpoints = endpoint_map(pred)
    gold_endpoints = endpoint_map(gold)
    comparable = set(pred_endpoints) & set(gold_endpoints)
    relation_matches = sum(
        len(pred_endpoints[pair] & gold_endpoints[pair]) for pair in comparable
    )
    comparable_pred = sum(len(pred_endpoints[pair]) for pair in comparable)

    added_nodes = set(g_nodes) - set(x_nodes)
    added_edges = set(g_edges) - set(x_edges)
    removed_edges = set(x_edges) - set(g_edges)
    node_add_recall = ratio(len(added_nodes & set(p_nodes)), len(added_nodes))
    edge_add_recall = ratio(len(added_edges & set(p_edges)), len(added_edges))
    additions_needed = len(added_nodes) + len(added_edges)
    additions_found = len(added_nodes & set(p_nodes)) + len(
        added_edges & set(p_edges)
    )
    addition_recall = ratio(additions_found, additions_needed)
    removal_rate = ratio(len(removed_edges - set(p_edges)), len(removed_edges))

    source_endpoints = endpoint_map(source)
    relation_repair_keys = {
        key
        for key, edge in g_edges.items()
        if (str(edge["source"]), str(edge["target"])) in source_endpoints
        and key not in x_edges
    }
    wrong_relation_keys = {
        key
        for key, edge in x_edges.items()
        if (str(edge["source"]), str(edge["target"])) in gold_endpoints
        and key not in g_edges
    }
    relation_repair_recall = ratio(
        len(relation_repair_keys & set(p_edges)), len(relation_repair_keys)
    )
    wrong_relation_removal_rate = ratio(
        len(wrong_relation_keys - set(p_edges)), len(wrong_relation_keys)
    )

    fixed_nodes = set(map(str, fixed_node_ids or []))
    fixed_edges = set(map(str, fixed_edge_keys or []))
    fixed_node_payload_exact = ratio(
        sum(
            node_id in p_nodes
            and node_id in x_nodes
            and p_nodes[node_id] == x_nodes[node_id]
            for node_id in fixed_nodes
        ),
        len(fixed_nodes),
    )
    fixed_edge_payload_exact = ratio(
        sum(
            key in p_edges and key in x_edges and p_edges[key] == x_edges[key]
            for key in fixed_edges
        ),
        len(fixed_edges),
    )

    metrics = {
        "node_precision": node_precision,
        "node_recall": node_recall,
        "node_f1": node_f1,
        "edge_precision": edge_precision,
        "edge_recall": edge_recall,
        "edge_f1": edge_f1,
        "relation_accuracy": ratio(
            relation_matches, comparable_pred, empty=0.0
        ),
        "relation_repair_recall": relation_repair_recall,
        "wrong_relation_removal_rate": wrong_relation_removal_rate,
        "node_add_recall": node_add_recall,
        "edge_add_recall": edge_add_recall,
        "addition_recall": addition_recall,
        "removal_rate": removal_rate,
        "fixed_object_exact": min(
            fixed_node_payload_exact, fixed_edge_payload_exact
        ),
        "spurious_nodes": float(len(set(p_nodes) - set(g_nodes))),
        "spurious_edges": float(len(set(p_edges) - set(g_edges))),
        "exact_canonical_match": float(
            pred == gold and raw_audit["structural_validity"] == 1.0
        ),
    }
    metrics.update(raw_audit)
    return metrics


def mode_primary_score(mode: str, metrics: Mapping[str, float]) -> float:
    if mode == "prior_empty":
        return (
            0.35 * metrics["node_f1"]
            + 0.45 * metrics["edge_f1"]
            + 0.20 * metrics["relation_accuracy"]
        )
    if mode == "fixed_nodes_only":
        return (
            0.25 * metrics["node_add_recall"]
            + 0.50 * metrics["edge_recall"]
            + 0.25 * metrics["edge_f1"]
        )
    if mode == "missing_edges":
        return (
            0.55 * metrics["edge_add_recall"]
            + 0.30 * metrics["edge_recall"]
            + 0.15 * metrics["edge_precision"]
        )
    if mode == "partial_subgraph":
        return (
            0.25 * metrics["node_add_recall"]
            + 0.35 * metrics["edge_add_recall"]
            + 0.40 * metrics["edge_f1"]
        )
    if mode == "wrong_relations":
        return (
            0.45 * metrics["relation_repair_recall"]
            + 0.35 * metrics["wrong_relation_removal_rate"]
            + 0.20 * metrics["edge_f1"]
        )
    if mode == "extra_edges":
        return (
            0.45 * metrics["removal_rate"]
            + 0.35 * metrics["edge_precision"]
            + 0.20 * metrics["edge_f1"]
        )
    raise ValueError(f"unsupported mode: {mode}")


## 5. Construct a controlled six-mode test suite

Every case has the same reference mechanism. Only the corruption changes, so
differences between modes are easy to interpret. Additive cases mark supplied
content fixed; relation-repair and edge-removal cases deliberately leave the
graph editable.


In [ ]:
CONDITION = (
    "Complete or repair a mechanism graph explaining how increasing humidity "
    "reduces the stiffness of a silk fibroin film: humidity increases water "
    "uptake, water uptake increases polymer-chain mobility through "
    "plasticization, and increased chain mobility decreases stiffness."
)

TARGET_GRAPH = {
    "nodes": [
        {"id": "Humidity"},
        {"id": "WaterUptake"},
        {"id": "ChainMobility"},
        {"id": "Stiffness"},
    ],
    "edges": [
        {
            "source": "Humidity",
            "relation": "increases",
            "target": "WaterUptake",
        },
        {
            "source": "WaterUptake",
            "relation": "increases",
            "target": "ChainMobility",
        },
        {
            "source": "ChainMobility",
            "relation": "decreases",
            "target": "Stiffness",
        },
    ],
}

ALL_NODES = TARGET_GRAPH["nodes"]
ALL_EDGES = TARGET_GRAPH["edges"]


def make_case(
    mode: str,
    source_graph: Mapping[str, Any],
    *,
    fixed_policy: str,
    expected_operation: str,
    condition: Optional[str] = None,
    target_graph: Optional[Mapping[str, Any]] = None,
) -> dict[str, Any]:
    if mode not in GRAPH_COMPLETION_MODES:
        raise ValueError(f"unknown mode: {mode}")
    if fixed_policy not in {"all", "none"}:
        raise ValueError("fixed_policy must be 'all' or 'none'")
    resolved_condition = str(condition or CONDITION).strip()
    source = canonicalize_graph(source_graph)
    target = canonicalize_graph(
        TARGET_GRAPH if target_graph is None else target_graph
    )
    for name, graph in (("source", source), ("target", target)):
        errors = validate_graph_schema(graph)
        audit = audit_raw_graph(graph)
        if errors or not audit["structural_validity"]:
            raise ValueError(f"invalid {name} graph: {errors or audit}")
    fixed_node_ids = (
        [str(node["id"]) for node in source["nodes"]]
        if fixed_policy == "all"
        else []
    )
    fixed_edge_keys = (
        [edge_key(edge) for edge in source["edges"]]
        if fixed_policy == "all"
        else []
    )
    canvas = render_graph_canvas(
        source,
        fixed_node_ids=fixed_node_ids,
        fixed_edge_keys=fixed_edge_keys,
    )
    user_prompt = build_graph_completion_user_prompt(
        canvas, condition=resolved_condition, mode=mode
    )
    return {
        "mode": mode,
        "condition": resolved_condition,
        "source_graph": source,
        "target_graph": target,
        "fixed_policy": fixed_policy,
        "fixed_node_ids": fixed_node_ids,
        "fixed_edge_keys": fixed_edge_keys,
        "canvas": canvas,
        "user_prompt": user_prompt,
        "expected_operation": expected_operation,
    }


CASES = [
    make_case(
        "prior_empty",
        {"nodes": [], "edges": []},
        fixed_policy="all",
        expected_operation="Construct all nodes and edges from the condition.",
    ),
    make_case(
        "fixed_nodes_only",
        {"nodes": ALL_NODES, "edges": []},
        fixed_policy="all",
        expected_operation="Preserve all supplied nodes and infer the edges.",
    ),
    make_case(
        "missing_edges",
        {"nodes": ALL_NODES, "edges": [ALL_EDGES[0]]},
        fixed_policy="all",
        expected_operation="Preserve supplied objects and add two missing edges.",
    ),
    make_case(
        "partial_subgraph",
        {
            "nodes": [ALL_NODES[0], ALL_NODES[1], ALL_NODES[3]],
            "edges": [ALL_EDGES[0]],
        },
        fixed_policy="all",
        expected_operation="Add ChainMobility and complete the mechanism.",
    ),
    make_case(
        "wrong_relations",
        {
            "nodes": ALL_NODES,
            "edges": [
                {
                    "source": "Humidity",
                    "relation": "decreases",
                    "target": "WaterUptake",
                },
                {
                    "source": "WaterUptake",
                    "relation": "decreases",
                    "target": "ChainMobility",
                },
                {
                    "source": "ChainMobility",
                    "relation": "increases",
                    "target": "Stiffness",
                },
            ],
        },
        fixed_policy="none",
        expected_operation="Correct all three relation labels.",
    ),
    make_case(
        "extra_edges",
        {
            "nodes": ALL_NODES,
            "edges": ALL_EDGES
            + [
                {
                    "source": "Stiffness",
                    "relation": "increases",
                    "target": "Humidity",
                }
            ],
        },
        fixed_policy="none",
        expected_operation="Remove the unsupported Stiffness-to-Humidity edge.",
    ),
]

assert [case["mode"] for case in CASES] == list(GRAPH_COMPLETION_MODES)

display(
    pd.DataFrame(
        [
            {
                "mode": case["mode"],
                "fixed_policy": case["fixed_policy"],
                "input_nodes": len(case["source_graph"]["nodes"]),
                "input_edges": len(case["source_graph"]["edges"]),
                "expected_operation": case["expected_operation"],
            }
            for case in CASES
        ]
    )
)


The next cell displays the exact rendered canvases. `[FIXED]` appears only in
additive tasks where supplied objects must be preserved.


In [ ]:
for case in CASES:
    display(Markdown(f"### `{case['mode']}`"))
    print(case["canvas"])


## 6. Load the public model

This is the only large download. With `USE_4BIT=True`, weights are quantized
during loading. Set `USE_4BIT=False` in the configuration cell for BF16.


In [ ]:
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model_kwargs: dict[str, Any] = {
    "device_map": "auto",
    "low_cpu_mem_usage": True,
    "token": HF_TOKEN,
}
if USE_4BIT:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )
else:
    model_kwargs["dtype"] = COMPUTE_DTYPE

load_started = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.eval()
print(f"Loaded {MODEL_ID} in {time.perf_counter() - load_started:.1f} s")
print(f"Input device: {model.device}")


## 7. Standalone generation, scoring, and visualization helpers


In [ ]:
def generate_case(
    case: Mapping[str, Any],
    *,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
    max_new_tokens: int = MAX_NEW_TOKENS,
    seed: int = SEED,
) -> dict[str, Any]:
    """Generate one untouched completion and score it against the case target."""

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    prompt = apply_graph_completion_chat_template(
        tokenizer,
        str(case["user_prompt"]),
        enable_thinking=ENABLE_THINKING,
    )
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    )
    encoded = {key: value.to(model.device) for key, value in encoded.items()}
    prompt_tokens = int(encoded["input_ids"].shape[-1])
    generation_kwargs: dict[str, Any] = {
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0,
        "use_cache": True,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        generation_kwargs.update(
            {"temperature": temperature, "top_p": top_p}
        )
    started = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(**encoded, **generation_kwargs)
    elapsed = time.perf_counter() - started
    completion_ids = output_ids[0, prompt_tokens:]
    raw_completion = tokenizer.decode(
        completion_ids, skip_special_tokens=True
    )
    extraction = extract_final_answer(raw_completion)
    prediction = extraction.graph
    if prediction is None:
        metrics = {
            "node_precision": 0.0,
            "node_recall": 0.0,
            "node_f1": 0.0,
            "edge_precision": 0.0,
            "edge_recall": 0.0,
            "edge_f1": 0.0,
            "relation_accuracy": 0.0,
            "relation_repair_recall": 0.0,
            "wrong_relation_removal_rate": 0.0,
            "node_add_recall": 0.0,
            "edge_add_recall": 0.0,
            "addition_recall": 0.0,
            "removal_rate": 0.0,
            "fixed_object_exact": 0.0,
            "spurious_nodes": 0.0,
            "spurious_edges": 0.0,
            "schema_valid": 0.0,
            "duplicate_nodes": 0.0,
            "duplicate_edges": 0.0,
            "dangling_edges": 0.0,
            "structural_validity": 0.0,
            "exact_canonical_match": 0.0,
        }
    else:
        metrics = compute_graph_metrics(
            prediction,
            case["target_graph"],
            case["source_graph"],
            fixed_node_ids=list(case["fixed_node_ids"]),
            fixed_edge_keys=list(case["fixed_edge_keys"]),
        )
    metrics["mode_primary"] = mode_primary_score(str(case["mode"]), metrics)
    metrics["valid_completion"] = float(
        extraction.valid and metrics["structural_validity"] == 1.0
    )
    return {
        "mode": case["mode"],
        "expected_operation": case["expected_operation"],
        "fixed_policy": case["fixed_policy"],
        "prompt": prompt,
        "raw_completion": raw_completion,
        "prompt_token_count": prompt_tokens,
        "completion_token_count": int(completion_ids.numel()),
        "elapsed_seconds": elapsed,
        "extraction": asdict(extraction),
        "metrics": metrics,
        "source_graph": case["source_graph"],
        "target_graph": case["target_graph"],
        "fixed_node_ids": case["fixed_node_ids"],
        "fixed_edge_keys": case["fixed_edge_keys"],
    }


def display_raw_result(result: Mapping[str, Any]) -> None:
    """Show metrics plus collapsible prompt and untouched raw output."""

    metrics = result["metrics"]
    status = "✅ valid" if metrics["valid_completion"] else "❌ invalid"
    display(
        Markdown(
            f"## `{result['mode']}` — {status}\n\n"
            f"**Expected:** {result['expected_operation']}  \n"
            f"**Tokens:** {result['completion_token_count']}  \n"
            f"**Time:** {result['elapsed_seconds']:.1f} s"
        )
    )
    display(
        pd.DataFrame(
            [
                {
                    key: metrics[key]
                    for key in (
                        "valid_completion",
                        "exact_canonical_match",
                        "fixed_object_exact",
                        "node_f1",
                        "edge_f1",
                        "mode_primary",
                        "addition_recall",
                        "relation_repair_recall",
                        "removal_rate",
                    )
                }
            ]
        ).round(3)
    )
    errors = result["extraction"]["errors"]
    if errors:
        display(Markdown("**Extraction errors:** " + "; ".join(errors)))
    prompt_html = html.escape(str(result["prompt"]))
    output_html = html.escape(str(result["raw_completion"]))
    display(
        HTML(
            f"<details><summary><b>Exact effective prompt</b></summary>"
            f"<pre style='white-space:pre-wrap'>{prompt_html}</pre></details>"
            f"<details open><summary><b>Raw decoded model completion</b></summary>"
            f"<pre style='white-space:pre-wrap'>{output_html}</pre></details>"
        )
    )


def graph_as_networkx(graph: Mapping[str, Any]) -> nx.DiGraph:
    result = nx.DiGraph()
    for node in graph.get("nodes", []):
        result.add_node(str(node["id"]))
    for edge in graph.get("edges", []):
        result.add_edge(
            str(edge["source"]),
            str(edge["target"]),
            relation=str(edge["relation"]),
        )
    return result


def draw_graph(
    graph: Optional[Mapping[str, Any]],
    *,
    title: str,
    ax: plt.Axes,
    positions: Optional[dict[str, np.ndarray]] = None,
) -> None:
    if not graph or not graph.get("nodes"):
        ax.text(0.5, 0.5, "Empty / invalid graph", ha="center", va="center")
        ax.set_title(title)
        ax.axis("off")
        return
    network = graph_as_networkx(graph)
    if positions is None:
        positions = nx.spring_layout(network.to_undirected(), seed=7)
    positions = {
        node: positions.get(node, np.random.default_rng(7).random(2))
        for node in network.nodes
    }
    nx.draw_networkx_nodes(
        network, positions, ax=ax, node_color="#D9EAF7", edgecolors="#0072B2"
    )
    nx.draw_networkx_labels(network, positions, ax=ax, font_size=8)
    nx.draw_networkx_edges(
        network,
        positions,
        ax=ax,
        edge_color="#555555",
        arrows=True,
        arrowsize=15,
        connectionstyle="arc3,rad=0.05",
    )
    edge_labels = {
        (source, target): data.get("relation", "")
        for source, target, data in network.edges(data=True)
    }
    nx.draw_networkx_edge_labels(
        network, positions, edge_labels=edge_labels, ax=ax, font_size=7
    )
    ax.set_title(title)
    ax.axis("off")


def compare_case_graphs(result: Mapping[str, Any]) -> None:
    prediction = result["extraction"].get("canonical_graph")
    graphs = [result["source_graph"], prediction, result["target_graph"]]
    union = nx.Graph()
    for graph in graphs:
        if graph:
            union.add_nodes_from(str(node["id"]) for node in graph.get("nodes", []))
            union.add_edges_from(
                (str(edge["source"]), str(edge["target"]))
                for edge in graph.get("edges", [])
            )
    positions = nx.spring_layout(union, seed=7) if union.nodes else {}
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
    draw_graph(result["source_graph"], title="Input canvas", ax=axes[0], positions=positions)
    draw_graph(prediction, title="Model prediction", ax=axes[1], positions=positions)
    draw_graph(result["target_graph"], title="Reference graph", ax=axes[2], positions=positions)
    fig.suptitle(str(result["mode"]), fontsize=13, fontweight="bold")
    plt.show()


## 8. Run all six corruption/inpainting cases

Generation is sequential to keep Colab memory bounded. The cell prints every
raw completion exactly as decoded, scores it, and writes a durable JSONL after
each case. If a small GPU runs out of memory, reduce `MAX_NEW_TOKENS` in the
configuration cell and restart the runtime.


In [ ]:
RESULTS_PATH = Path("graph_preflexor_six_mode_results.jsonl")
RESULTS_PATH.write_text("", encoding="utf-8")
results: list[dict[str, Any]] = []

for case in tqdm(CASES, desc="Graph-inpainting cases", unit="case"):
    print(f"\nRunning {case['mode']} ...")
    result = generate_case(case)
    results.append(result)
    with RESULTS_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(result, ensure_ascii=False) + "\n")
    display_raw_result(result)

print(f"Saved {len(results)} raw/scored records to {RESULTS_PATH.resolve()}")


## 9. Compare performance across modes


In [ ]:
summary = pd.DataFrame(
    [
        {
            "mode": result["mode"],
            "valid": result["metrics"]["valid_completion"],
            "exact": result["metrics"]["exact_canonical_match"],
            "fixed": result["metrics"]["fixed_object_exact"],
            "node_f1": result["metrics"]["node_f1"],
            "edge_f1": result["metrics"]["edge_f1"],
            "mode_primary": result["metrics"]["mode_primary"],
            "completion_tokens": result["completion_token_count"],
            "seconds": result["elapsed_seconds"],
        }
        for result in results
    ]
)
display(summary.round(3))

plot_frame = summary.set_index("mode")[[
    "valid", "exact", "fixed", "node_f1", "edge_f1", "mode_primary"
]]
ax = plot_frame.plot(
    kind="bar",
    figsize=(13, 5.5),
    width=0.82,
    color=["#0072B2", "#E69F00", "#009E73", "#56B4E9", "#D55E00", "#CC79A7"],
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Exact symbolic metric")
ax.set_xlabel("Corruption / inpainting mode")
ax.set_title("Graph-PRefLexOR-4B-Inpainting: standalone six-mode test")
ax.grid(axis="y", alpha=0.25)
ax.legend(ncol=3, frameon=False)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig("graph_preflexor_six_mode_metrics.png", dpi=200, bbox_inches="tight")
plt.show()


## 10. Inspect input, prediction, and reference graphs

Each row below uses a shared layout so structural differences are visually
apparent.


In [ ]:
for result in results:
    compare_case_graphs(result)


## 11. Run a custom manual case

Edit the condition, graph, mode, and fixed policy below. Use `all` when the
supplied content must be preserved. Use `none` for repair/removal modes.


In [ ]:
RUN_CUSTOM_CASE = False

CUSTOM_CONDITION = (
    "Complete a mechanism graph explaining how temperature changes polymer "
    "stiffness through thermal motion and chain mobility."
)
CUSTOM_GRAPH = {
    "nodes": [
        {"id": "Temperature"},
        {"id": "ChainMobility"},
        {"id": "Stiffness"},
    ],
    "edges": [
        {
            "source": "Temperature",
            "relation": "increases",
            "target": "ChainMobility",
        }
    ],
}
CUSTOM_TARGET_GRAPH = {
    "nodes": [
        {"id": "Temperature"},
        {"id": "ChainMobility"},
        {"id": "Stiffness"},
    ],
    "edges": [
        {
            "source": "Temperature",
            "relation": "increases",
            "target": "ChainMobility",
        },
        {
            "source": "ChainMobility",
            "relation": "decreases",
            "target": "Stiffness",
        },
    ],
}
CUSTOM_MODE = "partial_subgraph"
CUSTOM_FIXED_POLICY = "all"

if RUN_CUSTOM_CASE:
    custom_case = make_case(
        CUSTOM_MODE,
        CUSTOM_GRAPH,
        fixed_policy=CUSTOM_FIXED_POLICY,
        expected_operation="User-defined graph completion",
        condition=CUSTOM_CONDITION,
        target_graph=CUSTOM_TARGET_GRAPH,
    )
    custom_result = generate_case(custom_case)
    display_raw_result(custom_result)
    compare_case_graphs(custom_result)
else:
    print("Set RUN_CUSTOM_CASE=True and rerun this cell to generate a custom case.")


## 12. Download results

The JSONL contains each exact prompt, raw decoded completion, parsed graph,
structural diagnostics, metrics, token counts, and runtime. The PNG contains
the six-mode metric summary.


In [ ]:
print(RESULTS_PATH.resolve())
print(Path("graph_preflexor_six_mode_metrics.png").resolve())

# Uncomment in Colab to download both files.
# from google.colab import files
# files.download(str(RESULTS_PATH))
# files.download("graph_preflexor_six_mode_metrics.png")


## Interpretation notes

- `valid` requires a complete `<answer>...</answer>` block, no trailing text,
  parseable JSON, the graph schema, and structural validity.
- `exact` requires the entire canonical graph to equal the reference.
- Node and edge array order does not affect canonical equality.
- Node IDs and `(source, relation, target)` edge identities are exact; there
  is no embedding-based synonym matching.
- `mode_primary` changes by corruption type and emphasizes construction,
  addition, relation repair, or removal as appropriate.
- These six controlled cases are an executable behavioral demonstration, not
  a statistically powered benchmark. Use the repository's official 3,641-task
  test workflow for publication-scale evaluation.
- Generated scientific graphs should be reviewed by a domain expert.
